# LABORATORIO 3: Modelo de Detección de Anomalías con ML
**Estudiante:** Mery Flores  
**Curso:** Seguridad en Redes

## Tarea 3.1 — Exploración y preprocesamiento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import joblib

# 1. Cargar el dataset
df = pd.read_csv('lab3/network_traffic.csv')
print("--- Estadísticas Descriptivas ---")
print(df.describe())

In [ ]:
# 2. Visualizar la distribución de bytes_sent y duration_sec con histogramas
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['bytes_sent'], bins=30, kde=True, color='blue')
plt.title('Distribución de Bytes Sent')

plt.subplot(1, 2, 2)
sns.histplot(df['duration_sec'], bins=30, kde=True, color='green')
plt.title('Distribución de Duration Sec')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Identificar y tratar valores nulos o atípicos extremos
print("Valores nulos por columna:")
print(df.isnull().sum())
df = df.dropna()  # Eliminar nulos si existieran

In [ ]:
# 4. Feature engineering: cree al menos 2 nuevas variables derivadas
df['ratio_bytes'] = df['bytes_sent'] / (df['bytes_recv'] + 1)
df['bytes_por_segundo'] = df['bytes_sent'] / (df['duration_sec'] + 1)
print(df[['ratio_bytes', 'bytes_por_segundo']].head())

In [ ]:
# 5. Normalice las features numéricas con StandardScaler
features_num = ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets', 'ratio_bytes', 'bytes_por_segundo']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features_num])

## Tarea 3.2 — Entrenamiento del modelo

In [ ]:
# 1. Entrenar Isolation Forest (excluyendo label)
clf = IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
clf.fit(X_scaled)

# 2. Obtener las predicciones (-1 = anomalía, 1 = normal)
df['pred'] = clf.predict(X_scaled)
df['true_label'] = df['label'].map({'normal': 1, 'anomaly': -1})

In [ ]:
# 3. Calcule las métricas de evaluación usando la columna label
precision = precision_score(df['true_label'], df['pred'], pos_label=-1)
recall = recall_score(df['true_label'], df['pred'], pos_label=-1)
f1 = f1_score(df['true_label'], df['pred'], pos_label=-1)
cm = confusion_matrix(df['true_label'], df['pred'])

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

In [ ]:
# 4. Grafique la matriz de confusión con seaborn
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Anomalía', 'Normal'], yticklabels=['Anomalía', 'Normal'])
plt.title('Matriz de Confusión')
plt.ylabel('Real')
plt.xlabel('Predicción')
plt.show()

## Tarea 3.3 — Interpretación y umbral dinámico

In [ ]:
# 1. Grafique el score de anomalía (decision_function)
df['anomaly_score'] = clf.decision_function(X_scaled)
plt.figure(figsize=(8,4))
sns.histplot(df['anomaly_score'], bins=50, color='purple', kde=True)
plt.title('Distribución de Scores de Anomalía')
plt.show()

In [ ]:
# 2. Curva de umbral vs F1 para validación dinámica
scores = df['anomaly_score']
thresholds = np.linspace(scores.min(), scores.max(), 100)
f1_scores = [f1_score(df['true_label'], np.where(scores < t, -1, 1), pos_label=-1) for t in thresholds]
best_thresh = thresholds[np.argmax(f1_scores)]

plt.figure(figsize=(8,4))
plt.plot(thresholds, f1_scores, color='orange', lw=2)
plt.axvline(best_thresh, color='blue', linestyle='--', label=f'Umbral óptimo: {best_thresh:.4f}')
plt.xlabel('Umbral (Score)')
plt.ylabel('F1-Score')
plt.title('Optimización de Umbral vs F1-Score')
plt.legend()
plt.show()

In [ ]:
# 3. Top 10 registros más anómalos
top10 = df.sort_values(by='anomaly_score').head(10)
print(top10[['src_ip', 'dst_ip', 'dst_port', 'bytes_sent', 'duration_sec', 'anomaly_score']])

### Análisis de Amenazas (Top 10):
Los registros identificados muestran comportamientos sumamente atípicos:
1. **Altos volúmenes de bytes enviados (`bytes_sent`)** combinados con una duración de conexión muy baja, lo cual es indicativo de un posible ataque de exfiltración masiva de datos por ráfagas.
2. **Conexiones repetitivas hacia puertos inusuales** que sugieren escaneo de puertos activo o actividad de Command & Control (C2).

## Tarea 3.4 — Exportación del modelo

In [ ]:
# Serializar componentes clave
joblib.dump(clf, 'lab3/modelo_anomalias.pkl')
joblib.dump(scaler, 'lab3/scaler.pkl')
print("Modelo y Scaler exportados con éxito en la carpeta lab3/")